# Lab 9 — Let an AI attack your project
### EP0418 · Topic 9 · 1 practical hour

---
Today you point an AI assistant at your own pipeline and ask it to **find what is
wrong with it**.

> **What is assessed is your rebuttal, not what the critic finds.**

The critic will raise things that are wrong. Saying *why* they are wrong, with
evidence, is worth exactly as much as fixing the ones that are right.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see a few lines saying *fetched ...* and nothing in red; that means the course files and tools are loaded. Every lab starts this way. **To write in a text box** like *Strangest number:* further down, double-click the box, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

# Run this first. The pre-lab cells need these files beside the notebook.
fetch("smart_building_sensors.csv", "smart_self_daily.csv", "smart_money_ohlcv.csv")

## Pre-lab A — Three checks your labs never used

Labs 1–8 all ran on classroom sensor data and a transport feed. Two important
checks never came up, because that data does not contain the problems they catch —
but **your project's data might**. Run this cell at home before the lab and read what happens.

In [ ]:
# --- zeros_are_real: is a zero a measurement, or a device that was off? -------
# In the building data, occupancy = 0 means the room was empty. A REAL zero.
b = pd.read_csv("smart_building_sensors.csv", parse_dates=["timestamp"])
check.reset()
check.zeros_are_real(b, "occupancy", max_zeros=20000)      # passes — real zeros

# In personal activity data, steps = 0 usually means the watch was in a drawer.
# Same value, completely different meaning.
if os.path.exists("smart_self_daily.csv"):
    s = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
    check.zeros_are_real(s, "steps")                       # NOTE — investigate

# --- no_missing: are there blanks left in what you are about to analyse? ------
check.no_missing(b, cols=["co2_ppm", "temp_c"])

check.report()

### Read those two `zeros_are_real` results

`occupancy = 0` **passed**. `steps = 0` came back as a **NOTE**.

The check cannot tell you which is which — you told it, by setting `max_zeros`.
An empty room really does have zero people in it. A person with a watch in a
drawer did not really take zero steps, and averaging those days in drags every
statistic you compute downward.

**If your project's data has a column where zero might mean "not recorded", decide now**
and write the decision into your CA3 report.

## Pre-lab B — Lookahead bias (Track 1, and anyone acting on a signal)

If you compute a signal from today's data and then act on today's data, you used
information you would not have had at the time. This is the single most common way
a student backtest produces an impressive and completely fake result.

In [ ]:
# TODO(you): only if your project acts on a signal. Otherwise read and move on.
if os.path.exists("smart_money_ohlcv.csv"):
    m = pd.read_csv("smart_money_ohlcv.csv", parse_dates=["Date"])
    raw_signal = (m.Close > m.Close.rolling(20).mean()).astype(int)

    check.reset()
    check.signal_is_shifted(raw_signal, raw_signal, by=1)        # FAILS - lookahead
    check.report()
    check.signal_is_shifted(raw_signal, raw_signal.shift(1), by=1)  # PASSES
    check.report()

The first call fails and the second passes, on the same data. The only difference
is one `.shift(1)` — acting on yesterday's signal instead of today's.

If your system triggers an action from a signal, **prove the shift is there.**

## Task 1 — Run the critic

Open a **new chat** for this, not the one that wrote your code: a fresh chat has
not seen your reasoning and has no work of its own to defend. Copy the prompt below
into ChatGPT, Claude, Copilot or Cursor, then paste in your own pipeline code: the
five stage functions and `pipeline()` from Lab 8. (In Colab, click in a cell,
Ctrl+A, Ctrl+C.)

---
```
You are reviewing a student data pipeline for a module on automated decision
systems. Be adversarial but specific — your job is to find real defects, not to
be encouraging. Flag only defects that affect correctness or the checks below;
do not suggest style changes or extra features.

Check for each of these and say PASS, FAIL or UNSURE with a one-line reason:

1. Silent row loss — does any step drop rows without accounting for them?
2. Sentinel values — are non-measurement codes (-999, 0, 9999) handled?
3. Zero vs missing — are zeros that mean "no reading" distinguished from real zeros?
4. Time index — is the data resampled to a regular grid before any .shift()?
5. Horizon mismatch — are the model and baseline scored at the same horizon?
6. Lookahead bias — does any signal use information from the future?
7. Baseline honesty — is the baseline the strongest one a sensible person would use?
8. Response layer — is the alert debounced and rate-limited?
9. Failure mode — what happens if the data source returns nothing?

For each FAIL give the specific line and what you would change.

Here is my pipeline:
[PASTE YOUR CODE]
```
---

## Task 2 — Record every finding, then answer it

Fill this in. **One row per finding.** Your verdict column is what gets marked.

| # | Probe | Critic's finding | My verdict | Evidence |
|---|---|---|---|---|
| 1 | Silent row loss | | Accept / Reject / Unsure | |
| 2 | Sentinel values | | Accept / Reject / Unsure | |
| 3 | Zero vs missing | | Accept / Reject / Unsure | |
| 4 | Time index | | Accept / Reject / Unsure | |
| 5 | Horizon mismatch | | Accept / Reject / Unsure | |
| 6 | Lookahead bias | | Accept / Reject / Unsure | |
| 7 | Baseline honesty | | Accept / Reject / Unsure | |
| 8 | Response layer | | Accept / Reject / Unsure | |
| 9 | Failure mode | | Accept / Reject / Unsure | |

**Accept** = it is right, and here is what I changed.
**Reject** = it is wrong, and here is the evidence that it is wrong.
**Unsure** = I could not tell; here is what I would need to check. A PASS from
the critic still gets a row: write "critic said PASS" and whether you agree.

A row that says "Reject — I checked rows 400-460 and the resample keeps the empty
slots as NaN, it does not drop them" is worth more than one that says "Accept".

## Task 3 — Fix what deserves fixing, then re-verify

For each finding you accepted, ask your assistant to fix it **and to test its fix**:

```
Fix this: [the finding]. Then write a test that shows the fix works:
print the row count before and after, and the smallest and largest
value of the columns it touches. Do not change any existing check.
```

Then judge its test, not only its answer. For each fix, write down:

- **What did the test actually check?** "It returns a table" proves nothing.
- **Did it change or remove any `check.` line?** It may not.
- **Did it show numbers, or only say "passed"?** A pass with no numbers is a claim, not evidence.

In [ ]:
import pandas as pd, numpy as np
import os
from ep0418 import check, show

# TODO(you): re-run your pipeline's checks after your fixes
...
check.report()

## The question the critic cannot answer

An AI critic can check your code. It cannot tell you **what happens on the days
your system is wrong, and who pays for it.** That question is the Week 14 record
box on your CA3 Build 3 sheet, and it goes straight into your CA3 reflection.
Think about it this week; write it next week.

---
## Read before you run

A Track 5 group asked the critic to check their weekly activity alert. The critic said PASS on “zero versus missing”. The code was:

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it.

1. **Read it. Do not run it.** Answer in the text box under the code cell
   (double-click it to type): first *what is wrong with it*, then *what it would
   produce and what it should produce*. The questions are just below.
2. **Ask Gemini to explain it.** Click once in the code cell. At its top right,
   click the icon that looks like a pencil with a small star (Colab calls it the
   Gemini icon) and choose **Explain code**. Read the explanation. Did it mention
   the fault? It often does not.
3. **Only then run it.** Delete the `# ` at the start of every line *below the
   first line* (the first line is the instruction and stays), then run the cell
   and see whether you were right.

Was the critic right? Eleven days in this file have `steps` equal to 0 because the watch was not worn. What do those days do to `weekly`, and to `df.steps.mean()`?

Write down how many days the rule flags with the zeros left in, and how many after the zeros are replaced with NaN (run `zeros_are_real` and then try both).

<details><summary>Hint 1: where to look</summary>

Inside `rolling().mean()`, is a 0 treated like a NaN, or like a reading?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Like a reading. A drawer day pulls the weekly average and the overall mean down. Flags: 28 days with the zeros in, 17 once they are NaN.

</details>

An explanation that reads well is not evidence the code is right; that is the
point of step 2. **To fix it afterwards:** click the same Gemini icon on the cell,
choose **Transform code**, and say what to change in plain words. Colab shows the
change as a diff before it is applied. Read the diff. If it touches a `check.` line,
reject it.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# df = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
# weekly = df.steps.rolling(7).mean()
# low_activity = weekly < 0.7 * df.steps.mean()

**What is wrong with it:**

**What it would produce, and what it should produce:**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: what an adversarial review is for; why a critic's PASS is not evidence; how to defend correct work against a wrong finding; why an agent may run checks but never judge them; the question the critic cannot answer: who pays when the system is wrong. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- The findings table is complete, with a verdict and evidence on **every** row.
- **Mistake caught** (below): the finding you rejected, and the evidence.

**Reality check:** if you accepted every finding without pushing back on any, you
probably have not read them closely enough. Critics are confidently wrong too —
that is the whole lesson of this module.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*